# Skincare product performance and customer satisfaction

Data: Kaggle "Sephora Products and Skincare Reviews" (`product_info.csv` and the `reviews_*.csv` files, scraped March 2023).
Put those files next to this notebook (or change `DATA_DIR`), then run the cells top to bottom.
SQL lives in `sql/`, everything the notebook writes goes to `outputs/`.

In [ ]:
import re
import sqlite3
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy.stats import spearmanr
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 50)

DATA_DIR = Path(".")          # folder with product_info.csv and the reviews_*.csv files
SQL_DIR = Path("sql")
OUT_DIR = Path("outputs")
DB_PATH = Path("sephora.db")
OUT_DIR.mkdir(exist_ok=True)

# The analysis window (reviews from 2018-01-01 on) is set inside sql/02 and sql/03,
# because is_recommended was only collected from 2018.
SNAPSHOT_DATE = pd.Timestamp("2023-03-21")      # last review date in the scrape
CORE_CATEGORIES = ["Moisturizers", "Treatments", "Cleansers", "Eye Care",
                   "Masks", "Sunscreen", "Lip Balms & Treatments", "Self Tanners"]
MODEL_CATEGORIES = [c for c in CORE_CATEGORIES if c != "Self Tanners"]   # Self Tanners: tiny and mixed (tools, removers)
Q4_YEARS = ("2018", "2022")                     # skin profile fields thin out in 2023
SHOW_SNIPPETS = False                           # True prints six example reviews per complaint theme

# Two listings whose reviews cannot be pinned to one product: DermalQuench and Liquid ExfoliKate
AMBIGUOUS_PRODUCTS = ["P306941", "P439927"]

## 1. Load and profile the raw data

In [ ]:
products = pd.read_csv(DATA_DIR / "product_info.csv")
reviews = pd.concat(
    (pd.read_csv(f, low_memory=False) for f in sorted(DATA_DIR.glob("reviews_*.csv"))),
    ignore_index=True,
)
reviews = reviews.drop(columns=["Unnamed: 0"])
reviews["submission_time"] = pd.to_datetime(reviews["submission_time"], errors="coerce")

print("products", products.shape, "| reviews", reviews.shape)
print(reviews.isnull().mean().round(3).sort_values(ascending=False).head(8))

# 1. date range and rating mix
print("\nDATE RANGE")
print(reviews["submission_time"].min(), "to", reviews["submission_time"].max())
print("\nRATING MIX")
print(reviews["rating"].value_counts(normalize=True).sort_index().round(3))

# 2. exact duplicate rows
print("\nDUPLICATES")
print("duplicate rows:", reviews.duplicated().sum())

# 3. which products do the reviews cover?
print("\nPRODUCT COVERAGE")
rev_ids = reviews["product_id"].unique()
print("unique products in product table:", products["product_id"].nunique())
print("products with reviews:", len(rev_ids))
matched = reviews["product_id"].isin(products["product_id"]).mean()
print("review rows matched to products:", round(matched * 100, 2), "%")
print("\nCategories represented in review data:")
print(products[products["product_id"].isin(rev_ids)]["primary_category"].value_counts())

# 4. does the review file hold all the reviews the site shows?
print("\nREVIEW COUNT COVERAGE")
counts_in_file = reviews.groupby("product_id").size().rename("reviews_in_file")
check = products.set_index("product_id")[["reviews"]].join(counts_in_file, how="inner")
check["coverage"] = check["reviews_in_file"] / check["reviews"]
print(check["coverage"].describe().round(2))
print(check.sort_values("coverage", ascending=False).head(10))
print((check["coverage"] > 1.05).sum(), "products with coverage > 1.05")

# 5. why are is_recommended and helpfulness missing?
print("\nIS_RECOMMENDED MISSINGNESS BY YEAR")
print(reviews.groupby(reviews["submission_time"].dt.year)["is_recommended"]
      .apply(lambda s: s.isnull().mean()).round(3))
print("\nHELPFULNESS MISSING WHEN NOBODY VOTED:",
      (reviews.loc[reviews["helpfulness"].isnull(), "total_feedback_count"] == 0).mean())

## 2. SQL layer

Raw tables go into SQLite, then the scripts in `sql/` build the clean tables:
`reviews_clean` (exact duplicates removed), `products_skincare`, and review stats for the 2018+ window.

In [ ]:
reviews_out = reviews.copy()
reviews_out["submission_time"] = reviews_out["submission_time"].dt.strftime("%Y-%m-%d")

conn = sqlite3.connect(DB_PATH)
products.to_sql("products_raw", conn, if_exists="replace", index=False)
reviews_out.to_sql("reviews_raw", conn, if_exists="replace", index=False, chunksize=100000)
del reviews_out

for script in ["01_clean_tables.sql", "02_product_stats.sql"]:
    conn.executescript((SQL_DIR / script).read_text())

print("reviews_clean rows:", pd.read_sql("SELECT COUNT(*) AS n FROM reviews_clean", conn)["n"][0])
print("products_skincare rows:", pd.read_sql("SELECT COUNT(*) AS n FROM products_skincare", conn)["n"][0])

# review volume and rating mix by year
print(pd.read_sql("""
SELECT substr(submission_time, 1, 4) AS yr,
       COUNT(*) AS reviews,
       ROUND(AVG(rating), 3) AS avg_rating,
       ROUND(AVG(rating = 5), 3) AS five_star_share,
       COUNT(DISTINCT product_id) AS products
FROM reviews_clean
GROUP BY yr
ORDER BY yr
""", conn))

# products where the review file holds more reviews than the site count
print(pd.read_sql("""
SELECT p.product_id, p.product_name, p.reviews AS site_reviews,
       c.n AS reviews_in_file, ROUND(1.0 * c.n / p.reviews, 2) AS coverage
FROM products_raw p
JOIN (SELECT product_id, COUNT(*) AS n FROM reviews_clean GROUP BY product_id) c
  USING (product_id)
WHERE p.reviews > 0 AND 1.0 * c.n / p.reviews > 1.05
ORDER BY coverage DESC
LIMIT 10
""", conn))

pa = pd.read_sql("SELECT * FROM product_analysis", conn)   # skincare products with 2018+ review stats

## 3. Reviews shared across listings

Sephora shows the same reviews on every size and edition of a product, and the scrape saved each copy.
Products that share (almost) all their reviews are linked into one family, and each shared review is kept once per family.

In [ ]:
win = pd.read_sql((SQL_DIR / "03_review_window.sql").read_text(), conn)

# one fingerprint per review: the same review posted under several listings gets the same key
win["key"] = pd.util.hash_pandas_object(
    win[["author_id", "submission_time", "review_title", "review_text"]].fillna(""),
    index=False,
)
print(f"{len(win):,} reviews in the window | {win['product_id'].nunique():,} products")

# how many reviews appear under more than one listing?
print(pd.read_sql("""
SELECT COUNT(*) AS duplicated_reviews, SUM(n_rows) AS rows_involved
FROM (
  SELECT author_id, submission_time, substr(review_text, 1, 80) AS txt,
         COUNT(*) AS n_rows
  FROM reviews_clean
  WHERE submission_time >= '2018-01-01'
  GROUP BY author_id, submission_time, txt
  HAVING COUNT(DISTINCT product_id) > 1
)
""", conn))

# product pairs that share at least one review
multi_key = (win[win["key"].duplicated(keep=False)][["key", "product_id"]]
             .drop_duplicates())
pairs = multi_key.merge(multi_key, on="key")
pairs = pairs[pairs["product_id_x"] < pairs["product_id_y"]]

shared = (pairs.groupby(["product_id_x", "product_id_y"]).size()
               .rename("shared").reset_index())
n_unique = win.groupby("product_id")["key"].nunique()
shared["n_x"] = shared["product_id_x"].map(n_unique)
shared["n_y"] = shared["product_id_y"].map(n_unique)
shared["overlap"] = shared["shared"] / shared[["n_x", "n_y"]].min(axis=1)
shared["kind"] = np.select(
    [shared["shared"].eq(shared["n_x"]) & shared["shared"].eq(shared["n_y"]),
     shared["overlap"].ge(0.999)],
    ["identical", "one inside the other"],
    default="partial",
)
print(len(shared), "product pairs share at least one review")
print(shared["kind"].value_counts())

# the overlap levels split cleanly: real variants sit near 1.0, coincidences near 0
big = shared[shared["shared"] >= 5]
print(pd.cut(big["overlap"], [0, .1, .25, .5, .75, .95, 1.0], include_lowest=True)
        .value_counts().sort_index())

# link identical or strongly overlapping pairs, then group linked products into families
links = shared[(shared["kind"] == "identical") |
               ((shared["shared"] >= 5) & (shared["overlap"] >= 0.5))]
ids = pd.Index(sorted(win["product_id"].unique()))
i = ids.get_indexer(links["product_id_x"])
j = ids.get_indexer(links["product_id_y"])
adj = coo_matrix((np.ones(len(links)), (i, j)), shape=(len(ids), len(ids)))
n_fam, labels = connected_components(adj, directed=False)

fam2 = pd.DataFrame({"product_id": ids, "family_id": labels})
fam2["family_size"] = fam2.groupby("family_id")["product_id"].transform("size")
print(len(links), "links |", len(fam2), "products ->", n_fam, "families")
print(fam2.drop_duplicates("family_id")["family_size"].value_counts().sort_index())

# keep one copy of every shared review within a family
w3 = win.merge(fam2[["product_id", "family_id"]], on="product_id").drop_duplicates(["family_id", "key"])
w3["date"] = pd.to_datetime(w3["submission_time"])
w3["yr"] = w3["submission_time"].str[:4]
print("reviews after collapsing:", len(w3))
print("review keys still shared across families:", w3["key"].duplicated().sum())

## 4. One row per product family

Each family is represented by its main listing: not a mini, not a set, not a limited edition, then the one with the most reviews.
Ratings are adjusted toward the average with an empirical Bayes shrinkage, so products with a handful of reviews do not top the rankings.

In [ ]:
meta = pa.set_index("product_id")[["product_name", "price_usd", "secondary_category", "limited_edition"]]
cand = fam2.join(meta, on="product_id")
cand["is_mini"] = (cand["secondary_category"].eq("Mini Size") |
                   cand["product_name"].str.contains(r"\bmini\b", case=False, na=False))
cand["is_set"] = cand["secondary_category"].eq("Value & Gift Sets")
cand["n_rev"] = cand["product_id"].map(win.groupby("product_id").size())

rep = (cand.sort_values(["family_id", "is_mini", "is_set", "limited_edition", "n_rev", "price_usd"],
                        ascending=[True, True, True, True, False, False])
           .drop_duplicates("family_id"))

# families with more than one full-size listing: the only place a wrong link could hide
non_mini = cand[~cand["is_mini"]]
cnt = non_mini.groupby("family_id").size()
multi_nm = cnt[cnt > 1].index
print(len(multi_nm), "families have more than one non-mini member")
print(cand[cand["family_id"].isin(multi_nm)]
      .sort_values(["family_id", "n_rev"], ascending=[True, False])
      [["family_id", "product_id", "product_name", "price_usd", "secondary_category", "n_rev"]])

# the two listings that cannot be told apart; found by product id, not by family number
ambiguous_families = fam2.loc[fam2["product_id"].isin(AMBIGUOUS_PRODUCTS), "family_id"].unique().tolist()
assert len(ambiguous_families) == 1, "the two ambiguous products should sit in one family"

alone = cand[(cand["family_size"] == 1) & cand["is_mini"]]
print(len(alone), "minis with no linked full size")

# rating stats from one copy of every review per family
g = w3.groupby("family_id")["rating"]
fs = pd.DataFrame({
    "n_reviews": g.size(),
    "avg_rating": g.mean(),
    "low_rating_share": g.apply(lambda s: (s <= 2).mean()),
    "five_star_share": g.apply(lambda s: (s == 5).mean()),
    "var": g.var(ddof=1),
})

# empirical Bayes shrinkage: m is how many reviews' worth of pull each product gets toward the mean
mu = w3["rating"].mean()
multi = fs.dropna(subset=["var"])
sigma2 = ((multi["n_reviews"] - 1) * multi["var"]).sum() / (multi["n_reviews"] - 1).sum()
tau2 = fs["avg_rating"].var() - (sigma2 / fs["n_reviews"]).mean()
m = sigma2 / tau2
print(f"overall mean {mu:.3f} | within var {sigma2:.3f} | between var {tau2:.3f} | m = {m:.1f}")

fam_tbl = (rep[["family_id", "product_id", "family_size", "is_mini"]]
           .rename(columns={"is_mini": "standalone_mini"})
           .merge(pa[["product_id", "product_name", "brand_name", "price_usd", "loves_count",
                      "secondary_category", "limited_edition", "is_new", "online_only",
                      "out_of_stock", "sephora_exclusive"]], on="product_id")
           .merge(fs.drop(columns="var").reset_index(), on="family_id"))

fam_tbl["adj_rating"] = (fam_tbl["n_reviews"] * fam_tbl["avg_rating"] + m * mu) / (fam_tbl["n_reviews"] + m)

# names are safe unless the family is ambiguous; standalone minis stay out of product comparisons
fam_tbl["name_ok"] = ~fam_tbl["family_id"].isin(ambiguous_families)
fam_tbl["comparable"] = fam_tbl["name_ok"] & ~fam_tbl["standalone_mini"]
fam_tbl["core_category"] = fam_tbl["secondary_category"].isin(CORE_CATEGORIES)

# the same adjustment toward each category's own average
fam_tbl["w"] = fam_tbl["n_reviews"] * fam_tbl["avg_rating"]
cat_mu = (fam_tbl.groupby("secondary_category")["w"].sum()
          / fam_tbl.groupby("secondary_category")["n_reviews"].sum())
fam_tbl["adj_rating_cat"] = ((fam_tbl["w"] + m * fam_tbl["secondary_category"].map(cat_mu))
                             / (fam_tbl["n_reviews"] + m))
fam_tbl = fam_tbl.drop(columns="w")
fam_tbl.to_sql("family_analysis", conn, if_exists="replace", index=False)

comp = fam_tbl[fam_tbl["comparable"]].copy()
print(len(fam_tbl), "families |", fam_tbl["standalone_mini"].sum(), "standalone minis |",
      len(comp), "comparable")
print(cat_mu.round(3))
for k_ in (30, 50, 100):
    print(f"comparable families with {k_}+ reviews:", (comp["n_reviews"] >= k_).sum())

## 5. Why small review counts need adjusting

In [ ]:
perfect = comp[comp["avg_rating"] == 5]
print(len(perfect), "families have a perfect 5.0 average | max reviews among them:",
      int(perfect["n_reviews"].max()), "| median:", perfect["n_reviews"].median())

comp["bucket"] = pd.cut(comp["n_reviews"], [0, 9, 19, 49, 99, 499, 100000],
                        labels=["<10", "10-19", "20-49", "50-99", "100-499", "500+"])
print(comp.groupby("bucket", observed=True).agg(
        families=("family_id", "size"),
        mean_avg=("avg_rating", "mean"),
        sd_avg=("avg_rating", "std"),
        pct_perfect=("avg_rating", lambda s: (s == 5).mean())).round(3))

# funnel plot: how far can an average drift from the overall mean by chance alone?
n_grid = np.logspace(0, np.log10(comp["n_reviews"].max()), 200)
se = np.sqrt(sigma2 / n_grid)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(comp["n_reviews"], comp["avg_rating"], s=12, alpha=0.35, color="#B47AA3")
ax.axhline(mu, color="#444", lw=1)
ax.plot(n_grid, mu + 1.96 * se, color="#444", ls="--", lw=1)
ax.plot(n_grid, mu - 1.96 * se, color="#444", ls="--", lw=1)
ax.set_xscale("log")
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{int(v):,}"))
ax.set_ylim(1, 5.05)
ax.set_xlabel("Reviews per product (log scale)")
ax.set_ylabel("Average rating")
ax.set_title("Few reviews, wild averages")
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
plt.tight_layout()
plt.savefig(OUT_DIR / "funnel_plot.png", dpi=200)
plt.show()

## 6. Price, satisfaction, and review activity

Products are compared inside comparison groups (secondary category, split by product type when a type has at least 25 products).

In [ ]:
terts = pd.read_sql("SELECT product_id, tertiary_category FROM products_skincare", conn)
ft = fam_tbl.merge(terts, on="product_id", how="left")

# family-level interest and time on the market
loves = (cand[["product_id", "family_id"]]
         .merge(pa[["product_id", "loves_count"]], on="product_id")
         .groupby("family_id")["loves_count"].sum().rename("family_loves"))
ft = ft.merge(loves, on="family_id", how="left")

first_review = w3.groupby("family_id")["date"].min().rename("first_review")
ft = ft.merge(first_review, on="family_id", how="left")
ft["months_listed"] = (SNAPSHOT_DATE - ft["first_review"]).dt.days / 30.4
ft["reviews_per_month"] = ft["n_reviews"] / ft["months_listed"].clip(lower=1)

# comparable core-category products with 50+ reviews and a known price
base = ft[ft["comparable"] & ft["secondary_category"].isin(MODEL_CATEGORIES)
          & (ft["n_reviews"] >= 50)].dropna(subset=["price_usd"]).copy()

k = (base.groupby(["secondary_category", "tertiary_category"]).size()
         .rename("k").reset_index())
base = base.merge(k, on=["secondary_category", "tertiary_category"], how="left")
use_tert = base["tertiary_category"].notna() & (base["k"] >= 25)
base["group"] = np.where(use_tert,
                         base["secondary_category"] + " / " + base["tertiary_category"],
                         base["secondary_category"])

# percentiles inside each group
base["price_pct"] = base.groupby("group")["price_usd"].rank(pct=True)
base["rating_pct"] = base.groupby("group")["adj_rating_cat"].rank(pct=True)
base["velocity_pct"] = base.groupby("group")["reviews_per_month"].rank(pct=True)
base["price_tier"] = pd.cut(base["price_pct"], [0, .25, .5, .75, 1.0],
                            labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"],
                            include_lowest=True)
base["rating_gap"] = base["adj_rating_cat"] - base.groupby("group")["adj_rating_cat"].transform("mean")

base["overpriced"] = ((base["price_pct"] >= .75) & (base["rating_pct"] <= .25)
                      & (base["n_reviews"] >= 100))
base["hidden_gem"] = ((base["rating_pct"] >= .75) & (base["velocity_pct"] <= .5)
                      & (base["months_listed"] >= 12))

# drop comparison groups too small for quartiles to mean anything
gsize = base.groupby("group")["product_id"].transform("size")
b2 = base[gsize >= 20].copy()
print(len(base), "->", len(b2), "products |",
      base["group"].nunique() - b2["group"].nunique(), "small groups dropped")
print(b2["group"].value_counts())

# does price go with satisfaction? intervals resample whole brands
def brand_boot(df, stat, n=500, seed=7):
    rng = np.random.default_rng(seed)
    parts = {b: g for b, g in df.groupby("brand_name")}
    names = np.array(list(parts))
    vals = []
    for _ in range(n):
        pick = rng.choice(names, size=len(names), replace=True)
        vals.append(stat(pd.concat([parts[b] for b in pick], ignore_index=True)))
    return np.percentile(vals, [2.5, 97.5])

rho = spearmanr(b2["price_pct"], b2["rating_pct"])[0]
lo, hi = brand_boot(b2, lambda s: spearmanr(s["price_pct"], s["rating_pct"])[0])
print(f"rank correlation {rho:.3f} | brand-level 95% CI {lo:.3f} to {hi:.3f} "
      f"| {b2['brand_name'].nunique()} brands")

price_gap = lambda s: (s.loc[s["price_tier"] == "Q4 highest", "rating_gap"].mean()
                       - s.loc[s["price_tier"] == "Q1 lowest", "rating_gap"].mean())
lo, hi = brand_boot(b2, price_gap)
print(f"top vs bottom price quartile, rating gap: {price_gap(b2):.3f} (95% CI {lo:.3f} to {hi:.3f})")

# hidden gems lean toward pricey products (pricey products collect reviews more slowly),
# so the review rate is also ranked inside price quartile
print("\nSHARE FLAGGED AS HIDDEN GEM, BY PRICE QUARTILE")
print(b2.groupby("price_tier", observed=True)["hidden_gem"].mean().round(3))
b2["velocity_pct_tier"] = (b2.groupby(["group", "price_tier"], observed=True)
                             ["reviews_per_month"].rank(pct=True))
b2["hidden_gem2"] = ((b2["rating_pct"] >= .75) & (b2["velocity_pct_tier"] <= .5)
                     & (b2["months_listed"] >= 12))
print("original gems:", int(b2["hidden_gem"].sum()),
      "| price-adjusted gems:", int(b2["hidden_gem2"].sum()),
      "| in both:", int((b2["hidden_gem"] & b2["hidden_gem2"]).sum()))

# popular products with weak ratings: review rate times the rating shortfall
b2["shortfall"] = b2.groupby("group")["adj_rating_cat"].transform("mean") - b2["adj_rating_cat"]
fix = b2[(b2["rating_pct"] <= .25) & (b2["velocity_pct"] >= .5)
         & (b2["n_reviews"] >= 100)].copy()
fix["impact"] = fix["reviews_per_month"] * fix["shortfall"]
print("fix-first candidates:", len(fix), "| premium-priced low-rated:", int(b2["overpriced"].sum()))

## 7. What goes with higher ratings

Weighted regression on product families with category fixed effects, so every coefficient is a difference in stars inside the same comparison group.
Standard errors are clustered by brand. All of it is correlational.

In [ ]:
flags = ["limited_edition", "is_new", "online_only", "sephora_exclusive", "out_of_stock"]
flags2 = [f for f in flags if f != "limited_edition"]    # limited_edition covers only 4 products
attrs2 = "price_pct + " + " + ".join(flags2) + " + log_months"

d = b2.copy()
d["log_months"] = np.log(d["months_listed"].clip(lower=1))
d[flags] = d[flags].astype(int)
print(d[flags].sum().rename("products with flag"), "of", len(d), "products\n")

def fit(formula, data):
    w = 1 / (sigma2 / data["n_reviews"] + tau2)
    codes = pd.factorize(data["brand_name"])[0]
    return smf.wls(formula, data=data, weights=w).fit(
        cov_type="cluster", cov_kwds={"groups": codes})

def show2(res):
    ci = res.conf_int()
    t = pd.DataFrame({"coef": res.params, "ci_low": ci[0], "ci_high": ci[1]})
    t = t[~t.index.str.startswith("C(group") & (t.index != "Intercept")]
    t.index = [i.split("[T.")[-1].rstrip("]") if "[T." in i else i for i in t.index]
    return t.round(3)

# model 1: price as a within-group percentile (0 = cheapest, 1 = priciest)
print("MODEL 1 (stars within category, brand-clustered 95% CI)")
print(show2(fit(f"avg_rating ~ {attrs2} + C(group)", d)))

# model 2: price quartiles against the cheapest quartile
quartile_formula = ("avg_rating ~ C(price_tier, Treatment(reference='Q1 lowest')) + "
                    + " + ".join(flags2) + " + log_months + C(group)")
print("\nMODEL 2 (price quartiles vs lowest)")
print(show2(fit(quartile_formula, d)).loc[["Q2", "Q3", "Q4 highest"]])

# how much of the spread between products does each block of variables explain?
def adj_r2(rhs, data=d):
    w = 1 / (sigma2 / data["n_reviews"] + tau2)
    return smf.wls(f"avg_rating ~ {rhs} + C(group)", data=data, weights=w).fit().rsquared_adj

print("\nADJUSTED R-SQUARED (category fixed effects always included)")
print(f"category only: {adj_r2('1'):.3f}")
for name, rhs in {"price": "price_pct", "age": "log_months",
                  "flags": " + ".join(flags2), "price + age": "price_pct + log_months",
                  "all attributes": attrs2,
                  "all attributes + brand (upper bound)": attrs2 + " + C(brand_name)"}.items():
    print(f"+ {name}: {adj_r2(rhs):.3f}")

# is the age effect just calendar time? outcome = ratings from 2022 on only
late = w3[w3["date"] >= "2022-01-01"]
gl = late.groupby("family_id")["rating"].agg(n_late="size", avg_late="mean")
d2 = d.merge(gl, on="family_id")
d2 = d2[d2["n_late"] >= 30].copy()
d2["n_reviews"] = d2["n_late"]          # weights follow the reviews behind the outcome
print(f"\nSAME MODEL, 2022+ REVIEWS ONLY ({len(d2)} products)")
print(show2(fit(f"avg_late ~ {attrs2} + C(group)", d2)))

# is_new is mostly age in disguise
print("\ncorrelation is_new vs log_months:", round(d["is_new"].corr(d["log_months"]), 2))
print("median months listed | is_new:", round(d.loc[d["is_new"] == 1, "months_listed"].median(), 1),
      "| others:", round(d.loc[d["is_new"] == 0, "months_listed"].median(), 1))
print("\nMODEL WITHOUT AGE")
print(show2(fit("avg_rating ~ price_pct + " + " + ".join(flags2) + " + C(group)", d)))

# plain on/off comparison for each flag
rows = []
for c in flags:
    on, off = d[d[c] == 1], d[d[c] == 0]
    rows.append({"flag": c, "n_on": len(on),
                 "gap_on": on["rating_gap"].mean(), "gap_off": off["rating_gap"].mean()})
print("\nAVERAGE RATING GAP VS GROUP, FLAG ON VS OFF")
print(pd.DataFrame(rows).round(3))

## 8. What unhappy customers complain about

Keyword themes on 1-2 star reviews, compared with a category-balanced sample of 4-5 star reviews (the "lift").
Negation is handled inside a short window, and curly apostrophes are normalised first.

In [ ]:
# review text for every collapsed review
rv = w3[["family_id", "rating"]].copy()
rv["txt"] = ((w3["review_title"].fillna("") + " " + w3["review_text"].fillna(""))
             .str.lower().str.replace("\u2019", "'").str.replace("\u2018", "'"))
rv = rv[rv["txt"].str.len() >= 20].reset_index(drop=True)

# negation words close before a match (inside the same thought) cancel it
neg = re.compile(r"\b(no|not|non|never|without|free of|zero|cannot|can't|won't|wouldn't|shouldn't|isn't|aren't|wasn't|weren't|didn't|did not|doesn't|does not|don't|do not|hasn't|haven't)\b(?:(?!\b(?:but|however|though|although|yet)\b)[^.!?,;:]){0,25}$")

def first_hit(p, t):
    for m in p.finditer(t):
        if not neg.search(t[max(0, m.start() - 35):m.start()]):
            return m
    return None

plain = {
 "no results": r"\bno (noticeable |visible |real |significant )?(difference|results?|change|effect|improvement)|(haven't|hasn't|have not|has not|never|can't|cannot|couldn't|could not|don't|do not|didn't|did not) (really )?(see|seen|noticed?|saw)( any| a| much of a| much| a single)?( noticeable| real| visible| significant)? (difference|results?|change|improvement|effect)|did(n't| not) (work|do anything|help)|does(n't| not) (work|do anything|help)|not effective|ineffective|useless|nothing happened|did (absolutely |literally |basically |really )?nothing|didn't (really |seem to )?do (much|anything|a thing)|did not (really |seem to )?do (much|anything)|didnot (work|see|notice)|(barely|hardly) (noticed?|saw|see)( any| a)? (difference|results?|change)",
 "reformulation/discontinued": r"reformulat|new formula|old formula|formula (changed|is different)|changed (the )?formula|discontinu",
 "hype/overrated": r"\bhype(d|s)?\b|overrated|over-?hyped|doesn't live up|not what (it|they) (claim|promise|advertis)|misleading",
 "quantity/size": r"1/2 full|\bhalf (full|empty)\b|barely (any|enough) product|(so|very) little product|\btiny (amount|bottle|jar|tube)\b|smaller than (expected|i expected|advertised|pictured|the picture)|ran out (in|after) (less|just|only|a (week|few|couple))|\bnot (a lot|much|enough) (of )?product\b",
 "absorption/application": r"(doesn't|does not|didn't|did not|won't|never|not) (really |fully |properly |well )?(absorb|sink in|soak in|blend|melt|spread)|\bsits? on (top of )?(my |the )?(skin|face)|hard to (blend|spread|rub in|apply)|(doesn't|didn't) rinse",
 "review credibility": r"(fake|paid|incentivi[sz]ed|biased|planted|seeded) (\w+ ){0,2}reviews?|reviews? (are|were|seem|seems|look|looks) (fake|paid|biased|suspicious|incentivi[sz]ed)|(don't|do not|wouldn't|would not|can't|cannot) trust (the )?(other |these |any |all )?(reviews?|ratings?)|(confused|puzzled|baffled|shocked|surprised) (by|about|at|over) (all )?(the )?(positive|good|great|glowing|5[ -]star|five[ -]star)|not sure (why|how) (this|it) (got|has|gets) (such )?(good|great|so many|high)|only reviewed",
}

guard = {
 "skin reaction": r"\bbroke (me |my \w+ |\w+ )?out\b|\bbreak(s|ing)? (me |my \w+ )?out\b|\bbreak ?outs? (all over|on my|around|after|every)|(made|makes|make|making) (me|my \w+) break ?out|(caused|cause|causes|causing|gave|gives|giving) (me )?(a |some |many |more |bad |terrible |horrible |severe |painful |cystic |huge |new )*(breakouts?|pimples|zits)|cystic|\brash\b|\bhives\b|allergic reaction|burn(ed|s|ing)? (my|me)|\bstung\b|\bstings\b|\bstinging\b|\bitch(y|ing)\b|swollen|irritat(ed|es|ing|ion)|painful (break ?outs?|pimples|bumps|rash)|\bhurts?\b|\bpainful\b",
 "made skin worse": r"(made|makes|make|making) (it|them|things|my \w+( \w+)?) (so much |much |way |even |a lot |a little |slightly )?worse|(skin|face|acne|complexion) (got|gotten|became|become|looked|looks) (so much |much |way |even |a lot |a bit |a little |noticeably )?worse|\bworsen(ed|s|ing)?\b",
 "dryness complaint": r"dried (out )?(my|skin|face|lips)|dries (out )?(my|skin|face|lips)|(made|makes|left|leaves) my (skin|face|lips|eyes) (feel )?(so |very |really |extra )?(dry|tight|flaky|stripped)|stripp(ed|ing) (my|the|skin)|(?<!fast )(?<!quick )(?<!slow )(?<!air )\bdrying\b",
 "scent complaint": r"smell(s|ed)? (so |really |very |way |too |a bit |kind of |like )?(bad|terrible|awful|horrible|strong|overpowering|chemical|weird|funny|fake|gross|off|medicinal|alcohol|plastic|sour|rotten|cheap)|(strong|overpowering|chemical|awful|terrible|horrible|weird|artificial|fake) (smell|scent|odor|fragrance)|too (strong|perfumed)|\bstinks?\b|\bodor\b|(can't|cannot|couldn't|could not|unable to|won't|wouldn't) (get past|stand|handle|stomach|tolerate|deal with) (the |that |this |its |a )?(smell|scent|fragrance|odor)|(smell|scent|fragrance|odor) (is|was) (just |so |really |very |way |too |absolutely |extremely )*(bad|awful|terrible|horrible|strong|overpowering|nauseating|offensive|unbearable|off-putting)|smell(s|ed)? like (a |an )?(fish|rotten|rancid|sulfur|vinegar|paint|gas|garbage|nail polish|chemicals?|bleach|sweat|feet|burnt|mildew|mold)",
 "texture complaint": r"(feel(s|ing)?|felt|leaves?|left|is|was|are|so|too|very|really|extremely|super|bit|little|kind of) (\w+ ){0,2}(sticky|tacky|greasy|gritty|slimy|cakey|watery|runny)|sticky (feel|residue|film|coating|finish|texture|layer)|\bpill(s|ing|ed)?\b|clump|\bfilm\b|residue",
 "white cast": r"white cast|\bashy\b|chalky|gr[ae]y cast|purple cast|\bpasty\b|white film",
 "streaky": r"\bstreak(y|s|ing)?\b",
 "packaging defect": r"\bleak(ed|s|ing|y)?\b|\b(cracked|broken|damaged|dented|crushed) (pump|lid|cap|bottle|jar|tube|container|packaging|seal|nozzle|dispenser)\b|\b(pump|nozzle|dispenser|lid|cap) (broke|stopped|doesn't|did not|was (broken|cracked|stuck|clogged))|\bclogged (pump|nozzle|dispenser|applicator)\b|\bbroke (off|open)\b|\bspill(ed|s)?\b|(arrived|came|received) (open|empty|damaged|broken|leaking|half)|\bunsealed\b|\btamper|(seal|safety seal) (was )?(broken|missing)",
 "value for money": r"waste of (my )?(money|\$)|overpriced|too (expensive|pricey)|so (expensive|pricey)|way too much|rip ?off|not worth (the (price|money|cost)|my money|\$)|save your money|not worth it|not worth the investment|(doesn't|isn't|wasn't) worth (it|the (price|money|cost|investment)|buying|my money)",
 "spoilage/quality": r"\bmold(y|ed)?\b|\bmildew\b|\brancid\b|went bad|\bexpired\b|\bspoiled\b|\bseparat(ed|es|ing|ion)\b|\bcurdled\b|bacteria|\bfungus\b|contaminat",
}

# plain themes already carry their own negative wording; guarded themes skip negated matches
spec = {**{k: (re.compile(v), False) for k, v in plain.items()},
        **{k: (re.compile(v), True) for k, v in guard.items()}}
tcols = list(spec)

def flag(t, s):
    p, g = s
    return (first_hit(p, t) is not None) if g else (p.search(t) is not None)

def snippet(t, s, w=70):
    p, g = s
    m = first_hit(p, t) if g else p.search(t)
    return t[max(0, m.start() - w): m.end() + w].replace("\n", " ") if m else ""

# low-star reviews, and up to 40,000 high-star reviews per category (reweighted to the true mix)
rv2 = rv[["family_id", "rating", "txt"]].merge(
    fam_tbl[["family_id", "secondary_category"]], on="family_id", how="left")
low = rv2[rv2["rating"] <= 2].copy()
pool = rv2[(rv2["rating"] >= 4) & rv2["secondary_category"].notna()]
N = pool["secondary_category"].value_counts()
high = pd.concat([g.sample(min(len(g), 40000), random_state=1)
                  for _, g in pool.groupby("secondary_category")], ignore_index=True)
high["wt"] = high["secondary_category"].map(N / high["secondary_category"].value_counts())

for name, s in spec.items():
    low[name] = low["txt"].map(lambda t, s=s: flag(t, s))
    high[name] = high["txt"].map(lambda t, s=s: flag(t, s))
print(f"{len(low):,} low-star reviews | {len(high):,} baseline reviews")

# overall lift
hi_share = pd.Series({t: np.average(high[t].astype(float), weights=high["wt"]) for t in tcols})
theme_res = pd.DataFrame({"low_share": low[tcols].mean(), "high_share": hi_share,
                          "low_reviews": low[tcols].sum()})
theme_res["lift"] = theme_res["low_share"] / theme_res["high_share"]
print(theme_res.sort_values("lift", ascending=False).round(3))
print("low-star reviews matching no theme:", round(1 - low[tcols].any(axis=1).mean(), 3))

# read examples to audit a pattern
if SHOW_SNIPPETS:
    for t in tcols:
        ex = low.loc[low[t], "txt"]
        print("\n", t.upper(), f"({len(ex):,})")
        for x in ex.sample(min(6, len(ex)), random_state=3):
            print("   ...", snippet(x, spec[t]), "...")

# lift and raw share by category (blank where the evidence is thin)
big_cats = low["secondary_category"].value_counts()
big_cats = big_cats[big_cats >= 1000].index
lift_by, share_by = {}, {}
for c in big_cats:
    l, h = low[low["secondary_category"] == c], high[high["secondary_category"] == c]
    ok = (l[tcols].sum() >= 30) & (h[tcols].sum() >= 20)
    lift_by[c] = (l[tcols].mean() / h[tcols].mean()).where(ok)
    share_by[c] = l[tcols].mean()
lift_cat, share_cat = pd.DataFrame(lift_by), pd.DataFrame(share_by)
print("\nLIFT BY CATEGORY (blank = under 30 low-star or under 20 baseline matches)")
print(lift_cat.round(1))
print("\nSHARE OF LOW-STAR REVIEWS BY CATEGORY (%)")
print((share_cat * 100).round(1))

# do the fix-first products draw a different mix of complaints? products are the unit
fix_ids = set(fix["family_id"])
pl = low[low["secondary_category"].isin(set(fix["secondary_category"]))]
cnt = pl.groupby("family_id").size()
pl = pl[pl["family_id"].isin(cnt[cnt >= 20].index)]
pf = pl.groupby("family_id")[tcols].mean()
pf["fix"] = pf.index.isin(fix_ids)
print("\nproducts with 20+ low-star reviews:", pf["fix"].value_counts().to_dict())

A = pf[pf["fix"]][tcols].to_numpy(dtype=float)
B = pf[~pf["fix"]][tcols].to_numpy(dtype=float)
rng = np.random.default_rng(11)
boot = np.array([A[rng.integers(0, len(A), len(A))].mean(0) - B[rng.integers(0, len(B), len(B))].mean(0)
                 for _ in range(1000)])
theme_fix = pd.DataFrame({"fix_first": A.mean(0), "others": B.mean(0), "diff": A.mean(0) - B.mean(0),
                          "ci_low": np.percentile(boot, 2.5, axis=0),
                          "ci_high": np.percentile(boot, 97.5, axis=0)}, index=tcols)
print(theme_fix.sort_values("diff", ascending=False).round(3))

theme_res.to_csv(OUT_DIR / "theme_lift_overall.csv")
lift_cat.to_csv(OUT_DIR / "theme_lift_by_category.csv")
share_cat.to_csv(OUT_DIR / "theme_share_by_category.csv")
theme_fix.to_csv(OUT_DIR / "theme_fix_first.csv")

# long format for Tableau
for name, t in [("theme_lift_by_category", lift_cat), ("theme_share_by_category", share_cat)]:
    t.index.name = "theme"
    (t.reset_index().melt(id_vars="theme", var_name="category", value_name="value")
      .dropna(subset=["value"]).to_csv(OUT_DIR / f"{name}_long.csv", index=False))

## 9. Reviewers who say they got the product free

Some reviews say the product was free (influencer boxes, samples, "in exchange for my honest review").
Disclosure only counts when a reviewer wrote it down, so every share here is a floor.

In [ ]:
strict = re.compile(r"influenster|free ?product|sponsored|in exchange for|complimentary|courtesy of|\bgifted\b|vox ?box|bzz ?agent|for free|received (this|these|it|the \w+) (for free|free|as a gift|in exchange|from the brand)|in return for (my|an|a) (honest )?(review|opinion)|honest (review|opinion)[^.]{0,40}(free|gift|sample|received)|(free|gift|sample)[^.]{0,40}honest (review|opinion)")
not_inc = re.compile(r"\b(not|isn't|wasn't|never|no) [^.]{0,15}(sponsored|paid|gifted|free product|incentivi[sz]ed)|\bunpaid\b|\bunsponsored\b")
sample_pat = re.compile(r"\bsample\b|deluxe|play! by sephora|play box|subscription box|birchbox|ipsy|fabfitfun")

rv["disc"] = rv["txt"].map(lambda t: bool(strict.search(t)) and not not_inc.search(t))
rv["smp"] = rv["txt"].map(lambda t: bool(sample_pat.search(t)))

print("share disclosing a free product (strict):", round(rv["disc"].mean(), 3))
print("share mentioning a sample or subscription box:", round(rv["smp"].mean(), 3))
print(rv.groupby("rating")[["disc", "smp"]].mean().round(3))

# same product: disclosed vs undisclosed reviews
g = rv.groupby(["family_id", "disc"])["rating"].agg(["size", "mean"]).unstack("disc")
ok = (g[("size", True)] >= 20) & (g[("size", False)] >= 20)
dif = g.loc[ok, ("mean", True)] - g.loc[ok, ("mean", False)]
rng = np.random.default_rng(3)
bs = [dif.sample(len(dif), replace=True, random_state=int(rng.integers(1e9))).mean() for _ in range(1000)]
print(f"\n{ok.sum()} products with 20+ of each | rating gap, disclosed minus not: "
      f"mean {dif.mean():.3f}, median {dif.median():.3f}, "
      f"95% CI {np.percentile(bs, 2.5):.3f} to {np.percentile(bs, 97.5):.3f} (products resampled)")

# the same interval resampling whole brands
bn = fam_tbl.set_index("family_id")["brand_name"].reindex(dif.index)
gp = {b: s.to_numpy() for b, s in dif.groupby(bn)}
brand_names = np.array(list(gp))
rng = np.random.default_rng(5)
bs2 = [np.concatenate([gp[b] for b in rng.choice(brand_names, len(brand_names))]).mean() for _ in range(1000)]
print(f"gap {dif.mean():.3f} | brand-clustered 95% CI {np.percentile(bs2, 2.5):.3f} to "
      f"{np.percentile(bs2, 97.5):.3f} | {len(brand_names)} brands")

# do the hidden gems lean on disclosed reviews?
fam_inc = rv.groupby("family_id")["disc"].mean().rename("disc_share")
clean = rv[~rv["disc"]].groupby("family_id")["rating"].agg(n_clean="size", avg_clean="mean")
b3 = b2.merge(fam_inc, on="family_id", how="left").merge(clean, on="family_id", how="left")
print("\nmedian disclosed share | gems:", round(b3.loc[b3["hidden_gem2"], "disc_share"].median(), 3),
      "| others:", round(b3.loc[~b3["hidden_gem2"], "disc_share"].median(), 3))
print(b3.groupby("price_tier", observed=True)["disc_share"].median().round(3))

b3["adj_clean"] = (b3["n_clean"] * b3["avg_clean"] + m * b3["secondary_category"].map(cat_mu)) / (b3["n_clean"] + m)
ok3 = b3[b3["n_clean"] >= 30].copy()
ok3["clean_pct"] = ok3.groupby("group")["adj_clean"].rank(pct=True)
gems = ok3[ok3["hidden_gem2"]]
print(f"\n{len(gems)} of {int(b3['hidden_gem2'].sum())} gems have 30+ undisclosed reviews | "
      f"still top quartile on undisclosed reviews: {(gems['clean_pct'] >= .75).mean():.0%} | "
      f"average rating drop: {(gems['avg_rating'] - gems['avg_clean']).mean():.3f}")

# the price model again: with the disclosed share added, and on undisclosed reviews only
d3 = d.merge(fam_inc, on="family_id", how="left")
d3["disc_share"] = d3["disc_share"].fillna(0)
print("\nMODEL WITH DISCLOSED SHARE ADDED")
print(show2(fit(f"avg_rating ~ {attrs2} + disc_share + C(group)", d3))
      .loc[["price_pct", "log_months", "disc_share"]])

d4 = d.merge(clean.reset_index(), on="family_id", how="left")
d4 = d4[d4["n_clean"] >= 30].copy()
d4["n_reviews"] = d4["n_clean"]
d4["avg_rating"] = d4["avg_clean"]
print(f"\n{len(d4)} products with 30+ undisclosed reviews")
print("PRICE AND AGE, UNDISCLOSED REVIEWS ONLY")
print(show2(fit(f"avg_rating ~ {attrs2} + C(group)", d4)))
print("\nPRICE QUARTILES VS LOWEST, UNDISCLOSED ONLY")
print(show2(fit(quartile_formula, d4)).loc[["Q2", "Q3", "Q4 highest"]])

# do the complaint lifts change if the baseline drops disclosed reviews?
high["disc"] = high["txt"].map(lambda t: bool(strict.search(t)) and not not_inc.search(t))
hi_clean = high[~high["disc"]]
hs = pd.Series({t: np.average(hi_clean[t].astype(float), weights=hi_clean["wt"]) for t in tcols})
print("\nTHEME LIFT, ALL VS UNDISCLOSED BASELINE")
print(pd.DataFrame({"lift_all": theme_res["lift"],
                    "lift_undisclosed_baseline": theme_res["low_share"] / hs}).round(1))

## 10. Skin type and skin tone

Same-product comparisons (a fixed effect per product family), 2018-2022 only, because the profile fields thin out in 2023.
Skin type and tone are self-reported, optional, and skew light.

In [ ]:
print(w3[["skin_type", "skin_tone"]].isnull().groupby(w3["yr"]).mean().round(3))

# analysis set: 2018-2022 reviews from comparable families
fmeta = fam_tbl[["family_id", "brand_name", "secondary_category", "comparable"]]
w5 = (w3[w3["yr"].between(*Q4_YEARS)]
        .merge(fmeta, on="family_id", how="left"))
w5 = w5[w5["comparable"]].reset_index(drop=True)
print(len(w5), "reviews in the analysis set")

tone_map = {"porcelain": "1 fair/light", "fair": "1 fair/light", "fairLight": "1 fair/light",
            "light": "1 fair/light",
            "lightMedium": "2 light-medium/medium", "medium": "2 light-medium/medium",
            "mediumTan": "3 tan", "tan": "3 tan",
            "deep": "4 deep/rich/dark", "rich": "4 deep/rich/dark",
            "dark": "4 deep/rich/dark", "ebony": "4 deep/rich/dark"}   # olive and notSureST are left out
w5["tone_grp"] = w5["skin_tone"].map(tone_map)

# each repeat reviewer's usual rating, from their other reviews
an = w5.groupby("author_id")["rating"].transform("size")
asum = w5.groupby("author_id")["rating"].transform("sum")
w5["n_author"] = an
w5["author_loo"] = np.where(an >= 2, (asum - w5["rating"]) / (an - 1).clip(lower=1), np.nan)

print("\n", w5["skin_type"].value_counts().to_string())
print("\n", w5["tone_grp"].value_counts().to_string())
print("\nshare of reviews from reviewers with 2+ reviews here:", round((an >= 2).mean(), 3))
nrev = w5.groupby("author_id").size()
st = w5.dropna(subset=["skin_type"]).groupby("author_id")["skin_type"].nunique()
st = st[nrev.reindex(st.index) >= 2]
print("reviewers (2+ reviews) whose profile shows more than one skin type:", round((st > 1).mean(), 3))

def within_fit(df, cat_col, ref, controls=(), author_ci=False):
    d_ = df.dropna(subset=[cat_col] + list(controls)).reset_index(drop=True)
    X = pd.get_dummies(d_[cat_col]).drop(columns=[ref]).astype(float)
    cats = list(X.columns)
    for c in controls:
        X[c] = d_[c].astype(float).values
    X["y"] = d_["rating"].astype(float).values
    Xd = X - X.groupby(d_["family_id"].values).transform("mean")
    mod = sm.OLS(Xd["y"], Xd[cats + list(controls)])
    r_b = mod.fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d_["brand_name"])[0]})
    ci = r_b.conf_int().loc[cats]
    out = pd.DataFrame({"coef": r_b.params[cats], "lo": ci[0], "hi": ci[1],
                        "n": [int((d_[cat_col] == c).sum()) for c in cats]})
    if author_ci:
        r_a = mod.fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d_["author_id"])[0]})
        ca = r_a.conf_int().loc[cats]
        out["lo_author"], out["hi_author"] = ca[0], ca[1]
    return out, len(d_), int(d_[cat_col].eq(ref).sum())

def fmt(t):
    return t.apply(lambda r: f"{r['coef']:+.3f} ({r['lo']:+.3f} to {r['hi']:+.3f})", axis=1)

def run(cat_col, ref, label):
    a, n_a, ref_a = within_fit(w5, cat_col, ref, author_ci=True)
    sub = w5[w5["n_author"] >= 2]
    b, n_b, _ = within_fit(sub, cat_col, ref)
    c, n_c, _ = within_fit(sub, cat_col, ref, controls=("author_loo",))
    print(f"\n{label}: rating difference vs {ref}, same product (stars; brand-clustered 95% CI)")
    print(pd.DataFrame({"all reviews": fmt(a), "repeat reviewers only": fmt(b),
                        "+ reviewer leniency control": fmt(c), "n reviews": a["n"]}).to_string())
    print("author-clustered 95% CI, all reviews:")
    print(a[["coef", "lo_author", "hi_author"]].round(3).to_string())
    print(f"reviews: {n_a:,} all | {n_b:,} repeat reviewers | reference group {ref_a:,}")
    a.to_csv(OUT_DIR / f"q4_{cat_col}.csv")

run("skin_type", "combination", "SKIN TYPE")
run("tone_grp", "1 fair/light", "SKIN TONE")

# sunscreens: star ratings and white-cast mentions by tone group
sun = w5[w5["secondary_category"] == "Sunscreen"].copy()
a, n_a, ref_n = within_fit(sun, "tone_grp", "1 fair/light")
print("\nSUNSCREENS ONLY: rating difference vs fair/light, same product")
print(fmt(a).to_frame("diff (brand-clustered 95% CI)").join(a["n"]).to_string())
print(f"{n_a:,} reviews | reference group {ref_n:,}")

sun["txt"] = ((sun["review_title"].fillna("") + " " + sun["review_text"].fillna(""))
              .str.lower().str.replace("\u2019", "'").str.replace("\u2018", "'"))
sun["cast"] = sun["txt"].map(lambda t: flag(t, spec["white cast"]))
tone_base = sun.dropna(subset=["tone_grp"])
t = tone_base.groupby("tone_grp").agg(reviews=("cast", "size"), cast_share=("cast", "mean"),
                                      low_star_share=("rating", lambda s: (s <= 2).mean()))
lowc = (tone_base[tone_base["rating"] <= 2].groupby("tone_grp")["cast"]
        .agg(low_star_reviews="size", cast_share_in_low_star="mean"))
print("\nWHITE CAST MENTIONS IN SUNSCREEN REVIEWS, BY TONE GROUP")
print(t.join(lowc).round(3).to_string())
t.join(lowc).to_csv(OUT_DIR / "q4_sunscreen_white_cast_by_tone.csv")

# skin type difference by category
rows = []
for c in MODEL_CATEGORIES:
    sub = w5[w5["secondary_category"] == c]
    tt, n, _ = within_fit(sub, "skin_type", "combination")
    for typ, r in tt.iterrows():
        rows.append({"category": c, "skin_type": typ, "diff": r["coef"],
                     "lo": r["lo"], "hi": r["hi"], "n": int(r["n"])})
cat_tab = pd.DataFrame(rows)
cat_tab["excludes_zero"] = (cat_tab["lo"] > 0) | (cat_tab["hi"] < 0)
print("\nSKIN TYPE DIFFERENCE VS COMBINATION, BY CATEGORY (same product, stars)")
print(cat_tab.round(3).to_string(index=False))
cat_tab.to_csv(OUT_DIR / "q4_skin_type_by_category.csv", index=False)

## 11. Exports and pipeline counts

In [ ]:
# one product-level table with every flag from the analysis
gem_cols = ["family_id", "group", "price_pct", "price_tier", "rating_pct", "velocity_pct",
            "rating_gap", "overpriced", "hidden_gem", "hidden_gem2", "shortfall",
            "disc_share", "n_clean", "avg_clean", "adj_clean"]
prod = (ft.merge(b3[gem_cols], on="family_id", how="left")
          .merge(ok3[["family_id", "clean_pct"]], on="family_id", how="left")
          .merge(fix[["family_id", "impact"]], on="family_id", how="left"))
prod["fix_first"] = prod["impact"].notna()
prod["hidden_gem2"] = prod["hidden_gem2"].fillna(False).astype(bool)
prod["overpriced"] = prod["overpriced"].fillna(False).astype(bool)
prod["gem_testable"] = prod["hidden_gem2"] & (prod["n_clean"] >= 30)
prod["gem_holds_undisclosed"] = prod["gem_testable"] & (prod["clean_pct"] >= 0.75)

keep = ["family_id", "product_id", "product_name", "brand_name", "secondary_category",
        "tertiary_category", "group", "price_usd", "price_pct", "price_tier", "n_reviews",
        "avg_rating", "adj_rating_cat", "rating_pct", "reviews_per_month", "months_listed",
        "family_loves", "is_new", "online_only", "sephora_exclusive", "out_of_stock",
        "disc_share", "hidden_gem2", "gem_testable", "gem_holds_undisclosed",
        "overpriced", "fix_first", "impact", "comparable", "name_ok"]
prod.to_csv(OUT_DIR / "product_level_all.csv", index=False)
prod[prod["group"].notna()][keep].to_csv(OUT_DIR / "product_analysis_set.csv", index=False)

print(len(prod), "families |", int(prod["group"].notna().sum()), "in the analysis set")
print("hidden gems:", int(prod["hidden_gem2"].sum()),
      "| testable:", int(prod["gem_testable"].sum()),
      "| hold on undisclosed reviews:", int(prod["gem_holds_undisclosed"].sum()),
      "| premium-priced low-rated:", int(prod["overpriced"].sum()),
      "| fix-first:", int(prod["fix_first"].sum()))

# regression results in one file
rows = []
for name, data, f in [
        ("all reviews", d, f"avg_rating ~ {attrs2} + C(group)"),
        ("all reviews + disclosed share", d3, f"avg_rating ~ {attrs2} + disc_share + C(group)"),
        ("undisclosed reviews only", d4, f"avg_rating ~ {attrs2} + C(group)")]:
    t = show2(fit(f, data)).reset_index().rename(columns={"index": "term"})
    t["model"], t["products"] = name, len(data)
    rows.append(t)
for name, data in [("all reviews", d), ("undisclosed reviews only", d4)]:
    t = (show2(fit(quartile_formula, data)).loc[["Q2", "Q3", "Q4 highest"]]
         .reset_index().rename(columns={"index": "term"}))
    t["model"], t["products"] = name + " (price quartiles)", len(data)
    rows.append(t)
q1 = pd.concat(rows, ignore_index=True)
q1.to_csv(OUT_DIR / "q1_model_results.csv", index=False)

# free-product summary
disc_rows = [
    ("share of reviews disclosing a free product (strict pattern)", rv["disc"].mean()),
    ("same-product rating gap, disclosed minus undisclosed", dif.mean()),
    ("brand-clustered 95% CI, low", np.percentile(bs2, 2.5)),
    ("brand-clustered 95% CI, high", np.percentile(bs2, 97.5)),
    ("products in the gap comparison", len(dif)),
    ("hidden gems testable on undisclosed reviews", len(gems)),
    ("share of those still top quartile on undisclosed reviews", (gems["clean_pct"] >= .75).mean()),
]
pd.DataFrame(disc_rows, columns=["metric", "value"]).to_csv(OUT_DIR / "seeding_summary.csv", index=False)
rv.groupby("rating")[["disc", "smp"]].mean().to_csv(OUT_DIR / "seeding_share_by_rating.csv")

# every count in the write-up traces to this table
counts = {
    "raw reviews": len(reviews),
    "reviews after removing exact duplicates (SQL)": int(pd.read_sql("SELECT COUNT(*) AS n FROM reviews_clean", conn)["n"][0]),
    "reviews from 2018-01-01 onward": len(win),
    "after collapsing reviews shared across listings": len(w3),
    "Q4 analysis set (2018-2022, comparable products)": len(w5),
    "products in product table": len(products),
    "skincare products": int(pd.read_sql("SELECT COUNT(*) AS n FROM products_skincare", conn)["n"][0]),
    "skincare products with reviews in window": int(win["product_id"].nunique()),
    "product families": int(fam_tbl["family_id"].nunique()),
    "comparable families": int(fam_tbl["comparable"].sum()),
    "families in price/rating analysis set": len(b2),
}
pd.DataFrame({"step": list(counts), "count": list(counts.values())}).to_csv(OUT_DIR / "pipeline_counts.csv", index=False)
print("\n", pd.DataFrame({"count": counts}).to_string())

print()
for f in sorted(OUT_DIR.iterdir()):
    print(f"{f.name:45s} {f.stat().st_size / 1e6:8.2f} MB")

In [ ]:
# compare this run with the earlier interactive run (counts exact, estimates to rounding)
def q1_coef(model, term):
    r = q1[(q1["model"] == model) & (q1["term"] == term)]
    return float(r["coef"].iloc[0]) if len(r) else float("nan")

reference = [
    ("raw reviews",                            1094411, counts["raw reviews"], 0),
    ("after SQL duplicate removal",            1094182, counts["reviews after removing exact duplicates (SQL)"], 0),
    ("reviews from 2018 on",                   900965,  counts["reviews from 2018-01-01 onward"], 0),
    ("after collapsing shared reviews",        791714,  counts["after collapsing reviews shared across listings"], 0),
    ("Q4 analysis set",                        737390,  counts["Q4 analysis set (2018-2022, comparable products)"], 0),
    ("products in product table",              8494,    counts["products in product table"], 0),
    ("skincare products",                      2420,    counts["skincare products"], 0),
    ("skincare products with reviews",         2351,    counts["skincare products with reviews in window"], 0),
    ("product families",                       2265,    counts["product families"], 0),
    ("comparable families",                    2158,    counts["comparable families"], 0),
    ("families in price/rating set",           1256,    counts["families in price/rating analysis set"], 0),
    ("hidden gems",                            98,      int(prod["hidden_gem2"].sum()), 0),
    ("hidden gems testable",                   87,      int(prod["gem_testable"].sum()), 0),
    ("hidden gems holding up",                 76,      int(prod["gem_holds_undisclosed"].sum()), 0),
    ("premium-priced low-rated",               53,      int(prod["overpriced"].sum()), 0),
    ("fix-first products",                     109,     int(prod["fix_first"].sum()), 0),
    ("price coefficient, all reviews",         0.199,   q1_coef("all reviews", "price_pct"), 0.002),
    ("age coefficient, all reviews",           -0.179,  q1_coef("all reviews", "log_months"), 0.002),
    ("price coefficient, undisclosed only",    0.178,   q1_coef("undisclosed reviews only", "price_pct"), 0.002),
    ("disclosed minus undisclosed gap",        0.219,   float(dif.mean()), 0.002),
    ("theme lift: no results",                 14.476,  float(theme_res.loc["no results", "lift"]), 0.05),
    ("theme lift: value for money",            20.229,  float(theme_res.loc["value for money", "lift"]), 0.05),
]
rows = [{"check": n, "earlier run": e, "this run": g,
         "status": "ok" if abs(e - g) <= tol else "CHECK"} for n, e, g, tol in reference]
check_table = pd.DataFrame(rows)
nice = lambda v: f"{v:,.0f}" if float(v).is_integer() and abs(v) >= 1 else f"{v:.3f}"
print(check_table.assign(**{"earlier run": check_table["earlier run"].map(nice),
                            "this run": check_table["this run"].map(nice)}).to_string(index=False))
print("\nall match" if (check_table["status"] == "ok").all() else "\nsome values differ, see CHECK rows")

In [ ]:
conn.close()